In [1]:
# =============================================================================
# Celda 1 — Imports y configuración global
# =============================================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import shap
import xgboost as xgb
import json
from pathlib import Path
from datetime import datetime

# Configuración de rutas
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = BASE_DIR / 'data' / 'processed'
MODELS_DIR = BASE_DIR / 'models' / 'artifacts'
FIGURES_DIR = BASE_DIR / 'reports' / 'figures'
EVAL_DIR = BASE_DIR / 'reports' / 'evaluations'

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# Constantes globales
DATE_PREFIX = '20260309'
RANDOM_STATE = 42
CLASS_NAMES = ['Auditada', 'Glosada', 'Devuelta']
CLASS_COLORS = ['#2196F3', '#FF9800', '#F44336']  # Azul, Naranja, Rojo

# Estilo de gráficos
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.figsize': (14, 7),
})

# Mapeo de nombres técnicos a nombres legibles (español) para la tesis
FEATURE_LABELS = {
    'num_dx_distintos': 'Diagnósticos distintos',
    'num_tipos_servicio': 'Tipos de servicio',
    'num_items_con_autorizacion': 'Ítems con autorización',
    'tasa_devolucion_nit': 'Tasa devolución prestador',
    'tasa_glosa_nit': 'Tasa glosa prestador',
    'log_num_items_con_autorizacion': 'Log ítems con autorización',
    'valor_factura': 'Valor factura (COP)',
    'log_valor_factura': 'Log valor factura',
    'dias_hasta_radicacion': 'Días hasta radicación',
    'ratio_valor_vs_nit': 'Ratio valor vs promedio NIT',
    'log_valor_x_tasa_glosa': 'Log valor × tasa glosa',
    'dias_x_log_valor': 'Días × log valor',
    'ratio_autorizacion': 'Ratio autorización',
    'desviacion_dias_nit': 'Desviación días vs NIT',
    'ratio_cups_items': 'Ratio CUPS/ítems',
    'is_extemporanea': 'Radicación extemporánea',
    'tiene_fev': 'Tiene FEV electrónica',
    'regimen_encoded': 'Régimen (Subs=0/Cont=1)',
    'tipo_contrato_encoded': 'Tipo contrato',
    'porcentaje_mayor_item': '% mayor ítem',
    'discrepancia_valor': 'Discrepancia valor',
    'num_items': 'Número de ítems',
    'num_cups_distintos': 'CUPS distintos',
    'total_facturas_nit': 'Total facturas NIT',
    'valor_promedio_nit': 'Valor promedio NIT',
}

print(f"shap {shap.__version__}")
print(f"xgboost {xgb.__version__}")
print(f"Directorio base: {BASE_DIR}")
print(f"✓ Configuración cargada")

shap 0.51.0
xgboost 3.2.0
Directorio base: D:\Users\jcardonr\Documents\eafit\Codigo
✓ Configuración cargada


# 06 — Explicabilidad con SHAP (Fase 5)

**Objetivo:** Proporcionar explicaciones auditables para las predicciones del modelo XGBoost,
facilitando la adopción por parte de los auditores médicos de Savia Salud EPS.

**Entregables:**
1. Summary plot SHAP global (importancia de características)
2. SHAP values por clase (Auditada, Glosada, Devuelta)
3. Waterfall plots para ejemplos representativos por clase
4. Dependence plots para features clave
5. Función `explain_prediction()` con factores de riesgo en lenguaje natural
6. Figuras guardadas en `reports/figures/` a 300 dpi

## 1. Carga del modelo XGBoost y datos de test

Cargamos el modelo XGBoost v1 (mejor modelo, F1-macro=0.749, AUC=0.908) y el conjunto de test
para calcular los SHAP values. Usamos una muestra del test para eficiencia computacional.

In [2]:
# =============================================================================
# Celda 2 — Carga de modelo XGBoost y datos de test
# =============================================================================

# Cargar lista de features
with open(DATA_DIR / f'{DATE_PREFIX}_feature_list.txt', 'r') as f:
    FEATURES = [line.strip() for line in f if line.strip()]

print(f"Features cargadas: {len(FEATURES)}")

# Cargar modelo XGBoost desde formato nativo JSON
model_path = MODELS_DIR / f'{DATE_PREFIX}_xgboost_v1.json'
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(str(model_path))
print(f"✓ Modelo cargado: {model_path.name}")

# Cargar conjunto de test
test_df = pd.read_parquet(DATA_DIR / f'{DATE_PREFIX}_test.parquet')
X_test = test_df[FEATURES]
y_test = test_df['target']

print(f"✓ Test set: {X_test.shape[0]:,} filas × {X_test.shape[1]} features")
print(f"  Distribución: {dict(y_test.value_counts().sort_index())}")

# Muestra estratificada para SHAP (2,000 facturas — suficiente para convergencia)
# TreeExplainer es exacto, la muestra es solo para eficiencia en visualización
SHAP_SAMPLE_SIZE = 2000
np.random.seed(RANDOM_STATE)

sample_indices = []
for cls in [0, 1, 2]:
    cls_indices = y_test[y_test == cls].index.tolist()
    n_sample = min(len(cls_indices), SHAP_SAMPLE_SIZE // 3)
    sample_indices.extend(np.random.choice(cls_indices, size=n_sample, replace=False))

np.random.shuffle(sample_indices)
X_sample = X_test.loc[sample_indices].reset_index(drop=True)
y_sample = y_test.loc[sample_indices].reset_index(drop=True)

print(f"\n✓ Muestra SHAP: {len(X_sample):,} facturas (estratificada)")
print(f"  Distribución muestra: {dict(y_sample.value_counts().sort_index())}")

# Verificar predicciones del modelo en la muestra
y_pred_sample = xgb_model.predict(X_sample)
y_proba_sample = xgb_model.predict_proba(X_sample)

from sklearn.metrics import f1_score
f1_check = f1_score(y_sample, y_pred_sample, average='macro')
print(f"  F1-macro en muestra: {f1_check:.4f} (referencia test completo: 0.7490)")

Features cargadas: 48


✓ Modelo cargado: 20260309_xgboost_v1.json
✓ Test set: 40,193 filas × 48 features
  Distribución: {0: np.int64(13723), 1: np.int64(16143), 2: np.int64(10327)}

✓ Muestra SHAP: 1,998 facturas (estratificada)
  Distribución muestra: {0: np.int64(666), 1: np.int64(666), 2: np.int64(666)}
  F1-macro en muestra: 0.7468 (referencia test completo: 0.7490)


## 2. Cálculo de SHAP values

Usamos `shap.TreeExplainer` — método exacto para modelos basados en árboles (XGBoost).
Para clasificación multiclase, SHAP retorna un array de shape `(n_samples, n_features, n_classes)`,
donde cada valor indica la contribución de cada feature a la probabilidad de cada clase.

In [3]:
# =============================================================================
# Celda 3 — Cálculo de SHAP values con TreeExplainer
# =============================================================================
import time

print("Calculando SHAP values con TreeExplainer...")
print(f"  Modelo: XGBoost v1 ({xgb_model.best_iteration} árboles)")
print(f"  Muestra: {len(X_sample):,} facturas × {len(FEATURES)} features")

t0 = time.time()

# TreeExplainer — método exacto para XGBoost
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer(X_sample)

elapsed = time.time() - t0
print(f"\n✓ SHAP values calculados en {elapsed:.1f}s")
print(f"  Shape: {shap_values.shape}")
print(f"  Base values shape: {shap_values.base_values.shape}")

# Para multiclase, shap_values.values tiene shape (n_samples, n_features, n_classes)
# shap_values.base_values tiene shape (n_samples, n_classes)
if len(shap_values.shape) == 3:
    print(f"  → Estructura multiclase confirmada: {shap_values.shape}")
    print(f"  → Clases: {CLASS_NAMES}")
    for i, cls in enumerate(CLASS_NAMES):
        sv = shap_values.values[:, :, i]
        print(f"    Clase {i} ({cls}): mean|SHAP| = {np.abs(sv).mean():.4f}")
else:
    print("  → Estructura inesperada — verificar versión de SHAP")

Calculando SHAP values con TreeExplainer...
  Modelo: XGBoost v1 (515 árboles)
  Muestra: 1,998 facturas × 48 features



✓ SHAP values calculados en 3.3s
  Shape: (1998, 48, 3)
  Base values shape: (1998, 3)
  → Estructura multiclase confirmada: (1998, 48, 3)
  → Clases: ['Auditada', 'Glosada', 'Devuelta']
    Clase 0 (Auditada): mean|SHAP| = 0.0609
    Clase 1 (Glosada): mean|SHAP| = 0.0522
    Clase 2 (Devuelta): mean|SHAP| = 0.1715


## 3. Summary Plot Global — Importancia SHAP

El summary plot muestra la importancia global de cada feature medida por el valor absoluto
medio de los SHAP values. A diferencia de la importancia por ganancia (Fase 4), SHAP mide
la contribución marginal real de cada feature a las predicciones.

In [4]:
# =============================================================================
# Celda 4 — Summary plot global: importancia SHAP (mean |SHAP|)
# =============================================================================

# Calcular importancia global: mean |SHAP| promediado sobre las 3 clases
mean_abs_shap = np.abs(shap_values.values).mean(axis=(0, 2))  # (n_features,)
importance_df = pd.DataFrame({
    'feature': FEATURES,
    'mean_abs_shap': mean_abs_shap,
}).sort_values('mean_abs_shap', ascending=True)

# Etiquetas legibles
importance_df['label'] = importance_df['feature'].map(
    lambda x: FEATURE_LABELS.get(x, x)
)

# --- Figura 1: Bar plot de importancia SHAP global (Top 20) ---
fig, ax = plt.subplots(figsize=(10, 10))
top20 = importance_df.tail(20)
ax.barh(top20['label'], top20['mean_abs_shap'], color='steelblue', edgecolor='white')
ax.set_xlabel('mean(|SHAP value|)')
ax.set_title('Top 20 Features — Importancia SHAP Global\n(XGBoost v1, promedio sobre 3 clases)',
             fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'shap_01_importancia_global.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Figura guardada: reports/figures/shap_01_importancia_global.png")

# Mostrar top 10 en texto
print("\nTop-10 features por importancia SHAP global:")
for i, row in enumerate(importance_df.tail(10).iloc[::-1].itertuples(), 1):
    print(f"  {i:2d}. {row.feature:<40s} {row.mean_abs_shap:.4f}")

✓ Figura guardada: reports/figures/shap_01_importancia_global.png

Top-10 features por importancia SHAP global:
   1. log_num_items_con_autorizacion           1.2284
   2. log_num_dx_distintos                     0.4209
   3. tiene_fev                                0.3828
   4. log_valor_x_tasa_glosa                   0.2939
   5. num_items_con_autorizacion               0.2339
   6. num_tipos_servicio                       0.1913
   7. tasa_devolucion_nit                      0.1900
   8. ratio_autorizacion                       0.1340
   9. tasa_glosa_nit                           0.1245
  10. log_total_facturas_nit                   0.1012


## 4. SHAP Beeswarm Plots por Clase

Los beeswarm plots muestran cómo cada feature impacta la predicción de cada clase.
Cada punto es una factura: posición horizontal = SHAP value (contribución), color = valor de la feature.

Esto permite a los auditores entender **qué factores empujan hacia cada estado**.

In [5]:
# =============================================================================
# Celda 5 — Beeswarm plots por clase (Top 15 features)
# =============================================================================

# --- Figura 2: Beeswarm plots por clase ---
fig, axes = plt.subplots(1, 3, figsize=(24, 10))

for i, (cls_name, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    # Crear objeto SHAP Explanation para esta clase
    shap_cls = shap.Explanation(
        values=shap_values.values[:, :, i],
        base_values=shap_values.base_values[:, i],
        data=X_sample.values,
        feature_names=FEATURES,
    )

    plt.sca(axes[i])
    shap.plots.beeswarm(shap_cls, max_display=15, show=False)
    axes[i].set_title(f'Clase {i} — {cls_name}', fontweight='bold', fontsize=13)
    axes[i].set_xlabel('SHAP value (impacto en predicción)')

fig.suptitle('SHAP Beeswarm Plots por Clase — XGBoost v1\n'
             'Rojo = valor alto de la feature, Azul = valor bajo',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'shap_02_beeswarm_por_clase.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Figura guardada: reports/figures/shap_02_beeswarm_por_clase.png")

# --- Hallazgos clave por clase ---
print("\nHallazgos SHAP por clase:")
for i, cls_name in enumerate(CLASS_NAMES):
    sv_cls = shap_values.values[:, :, i]
    top_feat_idx = np.argsort(np.abs(sv_cls).mean(axis=0))[-5:][::-1]
    print(f"\n  Clase {i} ({cls_name}) — Top 5 features:")
    for rank, idx in enumerate(top_feat_idx, 1):
        feat = FEATURES[idx]
        mean_shap = sv_cls[:, idx].mean()
        direction = 'favorece' if mean_shap > 0 else 'reduce'
        print(f"    {rank}. {feat:<35s} mean={mean_shap:+.4f} ({direction})")

✓ Figura guardada: reports/figures/shap_02_beeswarm_por_clase.png

Hallazgos SHAP por clase:

  Clase 0 (Auditada) — Top 5 features:
    1. log_num_dx_distintos                mean=-0.1535 (reduce)
    2. num_tipos_servicio                  mean=-0.0283 (reduce)
    3. tiene_fev                           mean=-0.2059 (reduce)
    4. log_valor_x_tasa_glosa              mean=+0.0347 (favorece)
    5. ratio_autorizacion                  mean=+0.0304 (favorece)

  Clase 1 (Glosada) — Top 5 features:
    1. log_valor_x_tasa_glosa              mean=-0.2894 (reduce)
    2. tasa_glosa_nit                      mean=-0.0858 (reduce)
    3. log_num_items_con_autorizacion      mean=+0.0424 (favorece)
    4. tasa_devolucion_nit                 mean=-0.0899 (reduce)
    5. log_num_dx_distintos                mean=-0.0589 (reduce)

  Clase 2 (Devuelta) — Top 5 features:
    1. log_num_items_con_autorizacion      mean=-2.9974 (reduce)
    2. tiene_fev                           mean=+0.9131 (favorece)


## 5. Waterfall Plots — Explicaciones individuales por clase

Los waterfall plots muestran cómo las features contribuyen a la predicción de una factura
específica. Seleccionamos un ejemplo representativo (correctamente clasificado) para cada clase:
- **Auditada:** factura sin objeciones
- **Glosada:** factura con inconsistencias detectadas
- **Devuelta:** factura con error de forma

In [6]:
# =============================================================================
# Celda 6 — Waterfall plots para ejemplos representativos por clase
# =============================================================================

def find_representative_example(y_true, y_pred, y_proba, target_class):
    """Encuentra una factura correctamente clasificada con alta confianza.
    
    Args:
        y_true: Etiquetas reales.
        y_pred: Predicciones.
        y_proba: Probabilidades predichas.
        target_class: Clase objetivo (0, 1, o 2).
    
    Returns:
        Índice del ejemplo representativo.
    """
    # Facturas correctamente clasificadas de esta clase
    correct_mask = (y_true == target_class) & (y_pred == target_class)
    correct_indices = np.where(correct_mask)[0]
    
    if len(correct_indices) == 0:
        correct_indices = np.where(y_true == target_class)[0]
    
    # Seleccionar con confianza en el percentil 75 (representativa, no extrema)
    confidences = y_proba[correct_indices, target_class]
    p75_idx = np.argsort(confidences)[int(len(confidences) * 0.75)]
    return correct_indices[p75_idx]


# --- Figura 3: Waterfall plots (1 por clase) ---
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

example_indices = {}
for i, cls_name in enumerate(CLASS_NAMES):
    idx = find_representative_example(y_sample, y_pred_sample, y_proba_sample, i)
    example_indices[cls_name] = idx
    
    # Crear Explanation para esta factura y esta clase
    shap_example = shap.Explanation(
        values=shap_values.values[idx, :, i],
        base_values=shap_values.base_values[idx, i],
        data=X_sample.iloc[idx].values,
        feature_names=FEATURES,
    )
    
    plt.sca(axes[i])
    shap.plots.waterfall(shap_example, max_display=12, show=False)
    
    prob = y_proba_sample[idx, i]
    axes[i].set_title(
        f'Clase {i} — {cls_name}\n'
        f'P({cls_name})={prob:.2f} | Real={CLASS_NAMES[y_sample.iloc[idx]]}',
        fontweight='bold', fontsize=12
    )

fig.suptitle('Waterfall Plots — Explicación de predicción individual por clase\n'
             'Cada barra muestra la contribución de una feature a la probabilidad de la clase',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'shap_03_waterfall_por_clase.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Figura guardada: reports/figures/shap_03_waterfall_por_clase.png")

# Mostrar detalles de los ejemplos seleccionados
print("\nEjemplos representativos seleccionados:")
for cls_name, idx in example_indices.items():
    prob = y_proba_sample[idx]
    print(f"\n  {cls_name} (idx={idx}):")
    print(f"    P(Auditada)={prob[0]:.3f} | P(Glosada)={prob[1]:.3f} | P(Devuelta)={prob[2]:.3f}")
    print(f"    Real: {CLASS_NAMES[y_sample.iloc[idx]]} | Pred: {CLASS_NAMES[y_pred_sample[idx]]}")

✓ Figura guardada: reports/figures/shap_03_waterfall_por_clase.png

Ejemplos representativos seleccionados:

  Auditada (idx=1705):
    P(Auditada)=0.929 | P(Glosada)=0.071 | P(Devuelta)=0.000
    Real: Auditada | Pred: Auditada

  Glosada (idx=1610):
    P(Auditada)=0.021 | P(Glosada)=0.979 | P(Devuelta)=0.000
    Real: Glosada | Pred: Glosada

  Devuelta (idx=764):
    P(Auditada)=0.000 | P(Glosada)=0.000 | P(Devuelta)=1.000
    Real: Devuelta | Pred: Devuelta


## 6. Dependence Plots — Relación feature-SHAP para features clave

Los dependence plots revelan la relación entre el valor de una feature y su impacto SHAP.
Seleccionamos las top-4 features más importantes para la clase Glosada (clase 1),
que es la métrica primaria de optimización del proyecto.

In [7]:
# =============================================================================
# Celda 7 — Dependence plots para top-4 features (clase Glosada)
# =============================================================================

# Top 4 features para clase Glosada (clase 1)
sv_glosada = shap_values.values[:, :, 1]
glosada_importance = np.abs(sv_glosada).mean(axis=0)
top4_glosada_idx = np.argsort(glosada_importance)[-4:][::-1]
top4_features = [FEATURES[i] for i in top4_glosada_idx]

print(f"Top 4 features para clase Glosada: {top4_features}")

# --- Figura 4: Dependence plots ---
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, feat_idx, feat_name in zip(axes.flat, top4_glosada_idx, top4_features):
    feat_values = X_sample[feat_name].values
    shap_feat = sv_glosada[:, feat_idx]
    
    # Color por clase real
    scatter = ax.scatter(
        feat_values, shap_feat,
        c=y_sample.values, cmap='Set1', alpha=0.4, s=8,
        vmin=0, vmax=2,
    )
    ax.axhline(y=0, color='grey', linestyle='--', alpha=0.5)
    
    label = FEATURE_LABELS.get(feat_name, feat_name)
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('SHAP value (Glosada)', fontsize=11)
    ax.set_title(f'{label}', fontweight='bold')
    ax.grid(alpha=0.2)

# Leyenda compartida
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=plt.cm.Set1(i/2), 
           markersize=8, label=cls)
    for i, cls in enumerate(CLASS_NAMES)
]
fig.legend(handles=legend_elements, loc='upper center', ncol=3, 
           fontsize=11, bbox_to_anchor=(0.5, 1.02))

fig.suptitle('Dependence Plots — Top 4 Features para Clase Glosada\n'
             'Relación entre valor de la feature y contribución SHAP',
             fontsize=14, fontweight='bold', y=1.06)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'shap_04_dependence_glosada.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Figura guardada: reports/figures/shap_04_dependence_glosada.png")

Top 4 features para clase Glosada: ['log_valor_x_tasa_glosa', 'tasa_glosa_nit', 'log_num_items_con_autorizacion', 'tasa_devolucion_nit']


✓ Figura guardada: reports/figures/shap_04_dependence_glosada.png


## 7. Función `explain_prediction()` — Factores de riesgo en lenguaje natural

Función reutilizable que toma los valores de features de una factura y retorna:
1. Predicción y probabilidades
2. Los 5 factores de riesgo principales con explicación en lenguaje natural
3. Causal probable según taxonomía Res. 2284/2023

Esta función será integrada en la API REST (Fase 6).

In [8]:
# =============================================================================
# Celda 8 — Función explain_prediction() con factores de riesgo
# =============================================================================

# Mapeo de features a descripciones de riesgo en lenguaje natural
# Cada entrada: (condición_valor_alto, texto_valor_alto, texto_valor_bajo)
RISK_DESCRIPTIONS = {
    'tasa_glosa_nit': (
        True,
        'Tasa histórica de glosas del prestador es alta ({val:.1%})',
        'Tasa histórica de glosas del prestador es baja ({val:.1%})',
    ),
    'tasa_devolucion_nit': (
        True,
        'Tasa histórica de devoluciones del prestador es alta ({val:.1%})',
        'Tasa histórica de devoluciones del prestador es baja ({val:.1%})',
    ),
    'num_dx_distintos': (
        True,
        'Alto número de diagnósticos distintos ({val:.0f}) — posible facturación compleja',
        'Bajo número de diagnósticos ({val:.0f})',
    ),
    'num_tipos_servicio': (
        True,
        'Alta diversidad de tipos de servicio ({val:.0f})',
        'Baja diversidad de tipos de servicio ({val:.0f})',
    ),
    'num_items_con_autorizacion': (
        True,
        'Muchos ítems con autorización previa ({val:.0f})',
        'Pocos ítems con autorización previa ({val:.0f})',
    ),
    'valor_factura': (
        True,
        'Valor de factura elevado (${val:,.0f} COP)',
        'Valor de factura bajo (${val:,.0f} COP)',
    ),
    'dias_hasta_radicacion': (
        True,
        'Radicación tardía ({val:.0f} días después de la prestación)',
        'Radicación oportuna ({val:.0f} días)',
    ),
    'is_extemporanea': (
        True,
        'Radicación extemporánea (>22 días hábiles) — causal DE16',
        'Radicación dentro de plazo legal',
    ),
    'ratio_valor_vs_nit': (
        True,
        'Valor {val:.1f}x por encima del promedio del prestador',
        'Valor {val:.1f}x respecto al promedio del prestador (dentro de rango)',
    ),
    'log_valor_x_tasa_glosa': (
        True,
        'Combinación de alto valor y alta tasa de glosa (riesgo compuesto)',
        'Bajo riesgo compuesto valor-glosa',
    ),
    'discrepancia_valor': (
        True,
        'Discrepancia de ${val:,.0f} entre valor facturado y suma de detalles',
        'Consistencia entre valor facturado y detalles',
    ),
    'ratio_autorizacion': (
        False,
        'Bajo ratio de autorización previa ({val:.1%})',
        'Alto ratio de autorización previa ({val:.1%})',
    ),
    'porcentaje_mayor_item': (
        True,
        'Un solo ítem concentra {val:.0%} del valor total de la factura',
        'Distribución equilibrada de valores entre ítems',
    ),
    'ratio_cups_items': (
        True,
        'Alta diversidad CUPS/ítems ({val:.2f}) — posible duplicidad FA0301',
        'Baja diversidad CUPS/ítems',
    ),
    'tiene_fev': (
        False,
        'Sin factura electrónica de venta (FEV)',
        'Tiene FEV electrónica',
    ),
    'desviacion_dias_nit': (
        True,
        'Días de radicación {val:.0f} por encima del promedio del prestador',
        'Días de radicación dentro del rango habitual del prestador',
    ),
}

# Mapeo de features a causales probables (Res. 2284/2023)
FEATURE_TO_CAUSAL = {
    'is_extemporanea': 'DE16',
    'dias_hasta_radicacion': 'DE16',
    'ratio_cups_items': 'FA0301',
    'discrepancia_valor': 'FA0201',
    'tasa_glosa_nit': 'SO',
    'num_items_con_autorizacion': 'SO0201',
    'ratio_autorizacion': 'SO0201',
    'porcentaje_mayor_item': 'TA0101',
    'valor_factura': 'TA0101',
}

# Acciones sugeridas por clase
CLASS_ACTIONS = {
    0: 'Fast-track: pago programado sin auditoría manual',
    1: 'Enviar a auditor médico especializado para revisión detallada',
    2: 'Notificar al prestador para corrección y re-radicación',
}


def explain_prediction(
    features_dict: dict,
    model: xgb.XGBClassifier,
    shap_explainer: shap.TreeExplainer,
    feature_names: list[str],
    top_n: int = 5,
) -> dict:
    """Genera explicación en lenguaje natural para una predicción de factura.
    
    Calcula los SHAP values para una factura individual y traduce los factores
    más influyentes a descripciones comprensibles para auditores médicos.
    
    Args:
        features_dict: Diccionario {nombre_feature: valor} con las 48 features.
        model: Modelo XGBoost entrenado.
        shap_explainer: TreeExplainer pre-calculado.
        feature_names: Lista ordenada de nombres de features.
        top_n: Número de factores de riesgo a retornar (default 5).
    
    Returns:
        dict con predicción, probabilidades, factores de riesgo y acción sugerida.
    """
    # Preparar input
    X_input = pd.DataFrame([features_dict])[feature_names]
    
    # Predicción
    y_pred = model.predict(X_input)[0]
    y_proba = model.predict_proba(X_input)[0]
    clase_predicha = int(y_pred)
    
    # SHAP values para la clase predicha
    sv = shap_explainer(X_input)
    shap_for_pred_class = sv.values[0, :, clase_predicha]
    
    # Ordenar features por |SHAP| descendente
    sorted_indices = np.argsort(np.abs(shap_for_pred_class))[::-1]
    
    # Generar factores de riesgo en lenguaje natural
    factores_riesgo = []
    causal_probable = None
    
    for idx in sorted_indices[:top_n]:
        feat = feature_names[idx]
        shap_val = shap_for_pred_class[idx]
        feat_val = X_input[feat].iloc[0]
        
        if feat in RISK_DESCRIPTIONS:
            is_high_risky, text_high, text_low = RISK_DESCRIPTIONS[feat]
            if shap_val > 0:
                text = text_high if is_high_risky else text_low
            else:
                text = text_low if is_high_risky else text_high
            
            try:
                description = text.format(val=feat_val)
            except (ValueError, KeyError):
                description = text
        else:
            label = FEATURE_LABELS.get(feat, feat)
            direction = 'alto' if shap_val > 0 else 'bajo'
            description = f"{label} = {feat_val:.2f} (impacto {direction})"
        
        factores_riesgo.append({
            'feature': feat,
            'shap_value': float(shap_val),
            'feature_value': float(feat_val),
            'descripcion': description,
        })
        
        if causal_probable is None and feat in FEATURE_TO_CAUSAL:
            causal_probable = FEATURE_TO_CAUSAL[feat]
    
    return {
        'estado_predicho': CLASS_NAMES[clase_predicha].lower(),
        'clase': clase_predicha,
        'confianza': float(y_proba[clase_predicha]),
        'probabilidades': {
            'auditada': float(y_proba[0]),
            'glosada': float(y_proba[1]),
            'devuelta': float(y_proba[2]),
        },
        'factores_riesgo': [f['descripcion'] for f in factores_riesgo],
        'factores_detalle': factores_riesgo,
        'causal_probable': causal_probable,
        'accion_sugerida': CLASS_ACTIONS[clase_predicha],
    }


print("✓ Función explain_prediction() definida")
print(f"  - {len(RISK_DESCRIPTIONS)} features con descripciones en lenguaje natural")
print(f"  - {len(FEATURE_TO_CAUSAL)} features mapeadas a causales Res. 2284")

✓ Función explain_prediction() definida
  - 16 features con descripciones en lenguaje natural
  - 9 features mapeadas a causales Res. 2284


## 8. Demo: Explicaciones para facturas de ejemplo

Ejecutamos `explain_prediction()` sobre los 3 ejemplos representativos seleccionados
en la sección de waterfall plots, simulando el output que la API retornará en producción.

In [9]:
# =============================================================================
# Celda 9 — Demo de explain_prediction() con ejemplos reales
# =============================================================================

print("=" * 80)
print("  DEMO: explain_prediction() — Simulación de respuesta API")
print("=" * 80)

for cls_name, idx in example_indices.items():
    features_dict = X_sample.iloc[idx].to_dict()
    
    explanation = explain_prediction(
        features_dict=features_dict,
        model=xgb_model,
        shap_explainer=explainer,
        feature_names=FEATURES,
        top_n=5,
    )
    
    print(f"\n{'─' * 70}")
    print(f"  Factura ejemplo: {cls_name} (muestra idx={idx})")
    print(f"{'─' * 70}")
    print(f"  Estado predicho:  {explanation['estado_predicho'].upper()}")
    print(f"  Confianza:        {explanation['confianza']:.1%}")
    print(f"  Probabilidades:   A={explanation['probabilidades']['auditada']:.3f} "
          f"| G={explanation['probabilidades']['glosada']:.3f} "
          f"| D={explanation['probabilidades']['devuelta']:.3f}")
    print(f"  Causal probable:  {explanation['causal_probable']}")
    print(f"  Acción sugerida:  {explanation['accion_sugerida']}")
    print(f"\n  Factores de riesgo:")
    for i, factor in enumerate(explanation['factores_riesgo'], 1):
        print(f"    {i}. {factor}")
    
    print(f"\n  Detalle SHAP:")
    for f in explanation['factores_detalle']:
        print(f"    {f['feature']:<35s} SHAP={f['shap_value']:+.4f}  val={f['feature_value']:.2f}")

print(f"\n{'=' * 80}")
print("  ✓ Demo completada — formato compatible con POST /v1/predict (Fase 6)")
print(f"{'=' * 80}")

  DEMO: explain_prediction() — Simulación de respuesta API

──────────────────────────────────────────────────────────────────────
  Factura ejemplo: Auditada (muestra idx=1705)
──────────────────────────────────────────────────────────────────────
  Estado predicho:  AUDITADA
  Confianza:        92.9%
  Probabilidades:   A=0.929 | G=0.071 | D=0.000
  Causal probable:  None
  Acción sugerida:  Fast-track: pago programado sin auditoría manual

  Factores de riesgo:
    1. Alta diversidad de tipos de servicio (1)
    2. Combinación de alto valor y alta tasa de glosa (riesgo compuesto)
    3. log_total_facturas_nit = 13.37 (impacto alto)
    4. avg_valor_item = 68480.00 (impacto alto)
    5. Valor 0.1x por encima del promedio del prestador

  Detalle SHAP:
    num_tipos_servicio                  SHAP=+0.3357  val=1.00
    log_valor_x_tasa_glosa              SHAP=+0.2752  val=70.15
    log_total_facturas_nit              SHAP=+0.2236  val=13.37
    avg_valor_item                      SHAP=

## 9. Exportación de artefactos y resumen

Guardamos los SHAP values calculados y la importancia global para uso en la API y la tesis.

In [10]:
# =============================================================================
# Celda 10 — Exportación de artefactos SHAP y resumen final
# =============================================================================

# 1. Guardar importancia SHAP global en JSON (para integración LaTeX y API)
shap_importance = {}
for i, cls_name in enumerate(CLASS_NAMES):
    sv_cls = shap_values.values[:, :, i]
    cls_importance = {
        FEATURES[j]: float(np.abs(sv_cls[:, j]).mean())
        for j in range(len(FEATURES))
    }
    cls_importance = dict(sorted(cls_importance.items(), key=lambda x: x[1], reverse=True))
    shap_importance[cls_name] = cls_importance

# Importancia global (promedio sobre clases)
global_importance = {
    feat: float(val) 
    for feat, val in zip(importance_df['feature'], importance_df['mean_abs_shap'])
}
global_importance = dict(sorted(global_importance.items(), key=lambda x: x[1], reverse=True))

shap_output = {
    'fecha_calculo': datetime.now().isoformat(),
    'modelo': f'{DATE_PREFIX}_xgboost_v1',
    'n_samples': len(X_sample),
    'n_features': len(FEATURES),
    'importancia_global': global_importance,
    'importancia_por_clase': shap_importance,
    'top10_global': list(global_importance.keys())[:10],
}

shap_json_path = EVAL_DIR / f'{DATE_PREFIX}_shap_importance.json'
with open(shap_json_path, 'w', encoding='utf-8') as f:
    json.dump(shap_output, f, indent=2, ensure_ascii=False)
print(f"✓ Importancia SHAP exportada: {shap_json_path}")

# 2. Guardar SHAP values como numpy (para uso futuro en API)
shap_arrays_path = EVAL_DIR / f'{DATE_PREFIX}_shap_values_sample.npz'
np.savez_compressed(
    shap_arrays_path,
    shap_values=shap_values.values,
    base_values=shap_values.base_values,
    feature_names=np.array(FEATURES),
    y_sample=y_sample.values,
)
print(f"✓ SHAP values exportados: {shap_arrays_path}")

# --- RESUMEN FINAL ---
print(f"\n{'=' * 70}")
print(f"  FASE 5 — EXPLICABILIDAD (XAI) — RESUMEN")
print(f"{'=' * 70}")
print(f"\n  Modelo analizado: XGBoost v1")
print(f"  Muestra SHAP: {len(X_sample):,} facturas (estratificada)")
print(f"  Método: TreeExplainer (exacto para modelos de árboles)")

print(f"\n  Figuras generadas (4):")
print(f"    1. shap_01_importancia_global.png")
print(f"    2. shap_02_beeswarm_por_clase.png")
print(f"    3. shap_03_waterfall_por_clase.png")
print(f"    4. shap_04_dependence_glosada.png")

print(f"\n  Artefactos exportados:")
print(f"    - {shap_json_path.name}")
print(f"    - {shap_arrays_path.name}")

print(f"\n  Top-5 features por importancia SHAP global:")
for i, (feat, val) in enumerate(list(global_importance.items())[:5], 1):
    label = FEATURE_LABELS.get(feat, feat)
    print(f"    {i}. {label:<35s} (mean|SHAP| = {val:.4f})")

print(f"\n  Función explain_prediction():")
print(f"    - {len(RISK_DESCRIPTIONS)} features con descripciones en lenguaje natural")
print(f"    - {len(FEATURE_TO_CAUSAL)} features mapeadas a causales Res. 2284/2023")
print(f"    - Output compatible con schema API POST /v1/predict")

print(f"\n{'=' * 70}")
print(f"  ✓ Fase 5 completada — Listo para Fase 6 (API REST)")
print(f"{'=' * 70}")

✓ Importancia SHAP exportada: D:\Users\jcardonr\Documents\eafit\Codigo\reports\evaluations\20260309_shap_importance.json
✓ SHAP values exportados: D:\Users\jcardonr\Documents\eafit\Codigo\reports\evaluations\20260309_shap_values_sample.npz

  FASE 5 — EXPLICABILIDAD (XAI) — RESUMEN

  Modelo analizado: XGBoost v1
  Muestra SHAP: 1,998 facturas (estratificada)
  Método: TreeExplainer (exacto para modelos de árboles)

  Figuras generadas (4):
    1. shap_01_importancia_global.png
    2. shap_02_beeswarm_por_clase.png
    3. shap_03_waterfall_por_clase.png
    4. shap_04_dependence_glosada.png

  Artefactos exportados:
    - 20260309_shap_importance.json
    - 20260309_shap_values_sample.npz

  Top-5 features por importancia SHAP global:
    1. Log ítems con autorización          (mean|SHAP| = 1.2284)
    2. log_num_dx_distintos                (mean|SHAP| = 0.4209)
    3. Tiene FEV electrónica               (mean|SHAP| = 0.3828)
    4. Log valor × tasa glosa              (mean|SHAP| = 0.2